In [9]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from reportlab.lib.pagesizes import A4
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Image, PageBreak, KeepTogether)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch

df = pd.read_csv("./data/iris.csv")
df = df.dropna()
df.columns = ['SepalLength', 'SepalWidth', 'PetalLength', 'PetalWidth', 'Species']

# Function for scatter+analysis
def scatter_and_analysis(x_col, y_col, hue="Species", plot_filename=None):
    plt.figure(figsize=(6, 5))
    sns.scatterplot(data=df, x=x_col, y=y_col, hue=hue, palette="deep", s=70)
    plt.title(f"Scatter Plot: {x_col} vs {y_col}")
    plt.tight_layout()

    if plot_filename:
        plt.savefig(plot_filename, dpi=150, bbox_inches="tight")
    plt.close()

    corr, _ = pearsonr(df[x_col], df[y_col])

    if abs(corr) > 0.8:
        relation = "a very strong"
    elif abs(corr) > 0.6:
        relation = "a strong"
    elif abs(corr) > 0.4:
        relation = "a moderate"
    elif abs(corr) > 0.2:
        relation = "a weak"
    else:
        relation = "no significant"

    direction = "positive" if corr > 0 else "negative"

    analysis_text = f"<b>Correlation:</b> {corr:.2f} ({relation} {direction} relationship).<br/>"
    analysis_text += "<b>Species-wise Observations:</b><br/>"

    for sp in df["Species"].unique():
        subset = df[df["Species"] == sp]
        analysis_text += f"- {sp}: mean {x_col}={subset[x_col].mean():.2f}, mean {y_col}={subset[y_col].mean():.2f}<br/>"

    return analysis_text

relationships = [
    ("SepalLength", "SepalWidth"),
    ("PetalLength", "PetalWidth"),
    ("SepalLength", "PetalLength"),
    ("SepalWidth", "PetalWidth"),
    ("SepalLength", "PetalWidth"),
    ("SepalWidth", "PetalLength"),
]

#plots
all_results = []
for i, (x_col, y_col) in enumerate(relationships, 1):
    filename = f"scatter_{i}.png"
    text = scatter_and_analysis(x_col, y_col, plot_filename=filename)
    all_results.append((x_col, y_col, text, filename))

#Report
pdf_path = "ScatterPlot_Report.pdf"
doc = SimpleDocTemplate(pdf_path, pagesize=A4, rightMargin=40, leftMargin=40, topMargin=40, bottomMargin=40)

styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name='CustomHeading', fontSize=14, leading=16, spaceAfter=10, textColor="#2E4053", bold=True))
styles.add(ParagraphStyle(name='Analysis', fontSize=11, leading=14, spaceAfter=8))

story = []

#TitlE
story.append(Paragraph("Scatter Plot Interpretation Report", styles['Title']))
story.append(Spacer(1, 30))


#each relatiON
for i, (x, y, text, img_file) in enumerate(all_results, 1):
    content = []

    # Sec Heading
    content.append(Paragraph(f"Relationship {i}: {x} vs {y}", styles['CustomHeading']))
    content.append(Spacer(1, 8))

    # ImG
    img = Image(img_file)
    img._restrictSize(5.5*inch, 3.5*inch)
    content.append(img)
    content.append(Spacer(1, 12))

    content.append(Paragraph(text, styles['Analysis']))
    content.append(Spacer(1, 20))

    story.append(KeepTogether(content))

doc.build(story)

print(f"\n PDF report saved as: {pdf_path}")




 PDF report saved as: ScatterPlot_Report.pdf
